# Lab B - Evaluate Agents with Gemini Enterprise Agent Platform Eval Tools

**GENAI164 - Lab B of 2.** Lab A (a separate notebook) covered build-time evaluation with ADK's own framework; this lab covers the platform's managed evaluation tools.

In this lab you evaluate an agent built with the Agent Development Kit (ADK) using the Gemini Enterprise Agent Platform (GEAP) managed evaluation tools, through the Vertex AI Gen AI Evaluation SDK (`client.evals`). You simulate multi-turn users against the agent, score the runs with predefined and custom metrics, and group the failures with Automatic Loss Analysis. You do this first for a local agent, then for one deployed to **Agent Runtime** (the platform's managed runtime, formerly Vertex AI Agent Engine; the SDK still uses `agent_engines`).

This notebook runs in the Vertex AI Workbench environment in your lab project. Authentication is automatic through the instance service account.

You work through two parts:
1. **Local agent eval with user simulation:** predefined and custom metrics, then Automatic Loss Analysis.
2. **Managed eval on a deployed agent:** a single managed evaluation run through the Agent Platform evaluation tools.


## Lab at a glance

You evaluate a travel-booking agent two ways with GEAP's Gen AI Evaluation tools: first the **in-notebook (undeployed)** agent, then the **deployed** agent. "Local" means a step runs in your notebook; "managed" means it runs server-side on Google Cloud. The eval capabilities are GEAP's in both parts; the only difference is *where* they run.

| Stage / step | What you do | Runs where | Key feature |
|---|---|---|---|
| Setup | You install the SDKs, set your project, and create the evaluation client. | your notebook | **Gen AI Evaluation SDK**: your single entry point to every GEAP eval feature below. |
| Build the agent | You define the travel concierge agent in the notebook. | your notebook | **Agent Development Kit (ADK)**: builds the agent you will evaluate. |
| Deploy *(optional)* | You package and deploy the agent so it runs as a hosted service. | managed (cloud) | **Agent Runtime**: hosts the agent for the managed evaluation in Part 2. |
| **Part 1 - evaluate the in-notebook agent (you drive each step)** | | | |
| Step 1 - Generate scenarios | You auto-create multi-turn test cases from the agent's description, instead of writing them by hand. | managed eval service | **Synthetic scenario generation**: gives you test cases when you have none. |
| Step 2 - Simulate conversations | You let a simulated user chat with your agent over several turns to produce real conversation traces. | your notebook (local) | **User Simulator**: plays the user, so you can test multi-turn behavior without real users. |
| Step 3 - Evaluate | You author two custom metrics, then score the traces with those plus GEAP's built-in multi-turn metrics. | managed eval service | **Predefined + custom registered metrics**: grade tool use, trajectory, task success, and your own criteria. |
| Step 4 - Loss analysis | You group the failing conversations into themes to see *why* the agent failed, not just that it did. | managed (global) | **Automatic Loss Analysis**: clusters failures into named problem categories. |
| **Part 2 - evaluate the deployed agent (GEAP runs it all)** | | | |
| Step 1 - Generate scenarios | You generate a fresh set of test scenarios for the managed run (same feature as Part 1). | managed eval service | **Synthetic scenario generation**: reused here for the deployed agent. |
| Step 2 - Managed eval run | You hand the scenarios and the deployed agent to GEAP; it simulates, scores, and clusters failures in one job. | managed (cloud) | **Managed evaluation run**: the Eval Management Service runs simulation, scoring, and loss analysis server-side. |

**The key contrast:** in Part 1 *you* generate the conversations and then score them, step by step; in Part 2 you hand the scenarios to the managed **Eval Management Service** and it simulates *and* scores in one run.


## Getting started

### Install the SDKs

You install two packages: the Vertex AI SDK, which includes the Gen AI Evaluation tools you use throughout the lab, and the ADK, which builds the agent. Run both cells below, then continue to the kernel restart.


In [1]:
%pip install --upgrade -q "google-cloud-aiplatform[adk,agent_engines,evaluation]==1.163.0"

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 24.0.0 requires cryptography<43,>=41.0.5, but you have cryptography 50.0.1 which is incompatible.
opentelemetry-exporter-prometheus 0.65b0 requires opentelemetry-sdk~=1.44.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q google-adk

Note: you may need to restart the kernel to use updated packages.


### Restart the kernel

The install above upgrades packages that the base environment already has loaded. Run the next cell to restart the kernel so the new versions load cleanly. The kernel restarts automatically (this is expected) -- wait a few seconds for it to come back, then continue from the cell below it. Do not re-run the install cell.

In [3]:
# Restart the kernel so the updated packages load cleanly. The kernel will
# restart automatically; wait for it to come back, then continue from the next cell.
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

/var/tmp/ipykernel_5105/911708489.py:5: DeprecationWarning: `IPython.Application` is only a re-export of `traitlets.config.application.Application`; import it from traitlets directly. Accessing it here triggers an import of `IPython.core.application`, which is no longer imported when IPython is -- import that module explicitly if you rely on that import happening, in particular if you also rely on other submodules being transitively imported as a side effect.
  IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

### Set your Google Cloud project

You set your project and region and create the Gen AI Evaluation client. The next cells read your project from the environment and fill in sensible defaults, so you usually do not edit anything. Confirm the printed project and staging bucket after you run them.

A note on regions: the agent's model, `gemini-3.5-flash`, is served only in the `global` region, so the agent's model calls route to `global`. The evaluation client runs in `us-central1`. Agent Runtime cannot deploy to `global`, so the deploy step also uses `us-central1`.


In [1]:
import os
import uuid

import vertexai
from vertexai import agent_engines  # noqa: F401  (loads the submodule so vertexai.agent_engines resolves)
from google.adk import Agent
from vertexai import Client, types

In [2]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
# Run the eval client in us-central1 and pass allow_cross_region_model=True on
# the eval calls; the agent's model (gemini-3.5-flash) is global-only.
LOCATION = "us-central1"  # @param {type: "string"}
# fmt: on

In [3]:
# fmt: off
AGENT_ENGINE_GCS_BUCKET = ""  # @param {type: "string"}  -- blank = gs://PROJECT_ID-agent-engine
EVAL_RUN_GCS_BUCKET = ""      # @param {type: "string"}  -- blank = same as the staging bucket
# fmt: on

# Default the buckets to a valid, project-derived name if left blank. The SDK
# creates the bucket if it does not exist; it just needs a valid name.
if not AGENT_ENGINE_GCS_BUCKET:
    AGENT_ENGINE_GCS_BUCKET = f"gs://{PROJECT_ID}-agent-engine"
if not EVAL_RUN_GCS_BUCKET:
    EVAL_RUN_GCS_BUCKET = AGENT_ENGINE_GCS_BUCKET

# Eval client runs in us-central1; the agent's model is global-only (see above).
# Use v1beta1 API version for preview features (e.g., Evaluation Management Service / Evaluation Runs).
client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options={"api_version": "v1beta1"},
)

# Use Vertex AI with the ADK. The agent's model gemini-3.5-flash is a global-only
# preview, so route the agent's model calls to 'global'; the eval client stays in us-central1.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
print(f"Project: {PROJECT_ID} | Eval client location: {LOCATION}")
print(f"Staging bucket: {AGENT_ENGINE_GCS_BUCKET}")


Project: qwiklabs-gcp-04-8bc33a081848 | Eval client location: us-central1
Staging bucket: gs://qwiklabs-gcp-04-8bc33a081848-agent-engine


/var/tmp/ipykernel_6450/3340863024.py:15: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = Client(


## Helper functions

The next cell defines four helpers you use later:
- `poll_evaluation_run` watches a managed evaluation job and prints its status until it finishes.
- `deploy_adk_agent` packages and deploys your local agent to Agent Runtime in `us-central1`.
- `show_full_row` prints every column of one dataset row (pass `max_chars` to cap long traces), since the `.head()` tables truncate wide cells such as conversation plans and traces.
- `show_conversation` prints one simulated conversation as a readable transcript (user turns, agent turns, and tool calls); you use it in Part 1, Step 2.

In [4]:
def poll_evaluation_run(evaluation_run, client):
    import time

    from IPython.display import clear_output

    start_time = time.time()
    current_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    terminal_states = ["SUCCEEDED", "FAILED", "CANCELLED"]

    while current_run.state not in terminal_states:
        clear_output(wait=True)
        elapsed = int(time.time() - start_time)
        print(
            f"Evaluation started at: {time.strftime('%H:%M:%S', time.localtime(start_time))}, Elapsed time: {elapsed} seconds"
        )
        current_run.show()

        time.sleep(5)
        current_run = client.evals.get_evaluation_run(name=evaluation_run.name)

    clear_output(wait=True)
    print(f"Evaluation job completed in {int(time.time() - start_time)} seconds.")
    final_result = client.evals.get_evaluation_run(
        name=evaluation_run.name, include_evaluation_items=True
    )
    final_result.show()

def deploy_adk_agent(agent, location):
    """Deploy agent to agent engine"""
    app = vertexai.agent_engines.AdkApp(
        agent=agent,
        app_name=agent.name,
    )

    ae_client = Client(project=PROJECT_ID, location=location)

    print(
        "Deploying the agent to Agent Engine. This can take up to 10 mins.", flush=True
    )
    remote_app = ae_client.agent_engines.create(
        agent=app,
        config={
            "display_name": agent.name,
            "staging_bucket": AGENT_ENGINE_GCS_BUCKET,
            "requirements": [
                "google-cloud-aiplatform[adk,agent_engines]",
                "pydantic",
                "cloudpickle",
            ],
            "env_vars": {
                "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
                # Agent uses gemini-3.5-flash (preview, global-only). Agent Engine
                # itself can't run in 'global', so override the deployed app's
                # location env var to route the model calls there.
                "GOOGLE_CLOUD_LOCATION": "global",
                "GOOGLE_GENAI_USE_VERTEXAI": "1",
            },
        },
    )

    return remote_app

def show_full_row(eval_df, i=0, max_chars=None):
    """Print every column of one row. Set max_chars to cap each column's output,
    which is useful for the long agent_data conversation trace in Step 2. The
    .head() table truncates wide cells; this shows one row, optionally limited."""
    import json
    if len(eval_df) == 0:
        print("(no rows to show)")
        return
    row = eval_df.iloc[i]
    print("=" * 80)
    print(f"FULL VIEW OF ROW {i} (all columns)")
    print("=" * 80)
    for col in eval_df.columns:
        val = row[col]
        if isinstance(val, (dict, list)):
            text = json.dumps(val, indent=2, default=str, ensure_ascii=False)
        else:
            text = str(val)
        if max_chars is not None and len(text) > max_chars:
            text = text[:max_chars] + f"\n... [truncated; {len(text):,} chars total]"
        print(f"\n----- {col} -----")
        print(text)

def show_conversation(eval_df, i=0, max_text=300):
    """Print one simulated conversation from the trace as a readable transcript
    (user turns, agent turns, and tool calls). Much shorter than dumping the raw
    agent_data. Set max_text to change how much of each message is shown."""
    if len(eval_df) == 0 or "agent_data" not in eval_df.columns:
        print("(no agent_data column; run inference first)")
        return
    ad = eval_df.iloc[i]["agent_data"]
    if hasattr(ad, "model_dump"):
        ad = ad.model_dump(exclude_none=True)
    if not isinstance(ad, dict):
        print("(agent_data is not in the expected format; use show_full_row to inspect it)")
        return

    def clip(v):
        s = str(v)
        return s if len(s) <= max_text else s[:max_text] + " ...[clipped]"

    print("=" * 80)
    print(f"CONVERSATION (row {i})")
    print("=" * 80)
    turns = ad.get("turns") or []
    for turn in turns:
        for event in (turn.get("events") or []):
            content = event.get("content") or {}
            role = content.get("role")
            author = event.get("author") or "agent"
            for part in (content.get("parts") or []):
                if part.get("text"):
                    who = "User" if role == "user" else author
                    print(f"\n[{who}] {clip(part['text'])}")
                elif part.get("function_call"):
                    fc = part["function_call"]
                    print(f"    -> tool call: {fc.get('name')}({clip(fc.get('args'))})")
                elif part.get("function_response"):
                    print(f"    <- tool result: {part['function_response'].get('name')}")
    if not turns:
        print("(no turns found; use show_full_row to inspect the raw agent_data)")


## Build the agent

You evaluate a travel concierge agent built with the ADK. It acts as a primary orchestrator that hands work to two sub-agents, a `flight_agent` and a `hotel_agent`, and it has six tools for searching and booking flights and hotels over a small in-memory dataset.

The flight and hotel data ships in `travel_data.py` next to this notebook; the agent cell imports it. The next cell defines the agent. 


In [6]:
# @title Local ADK Agent
from travel_data import FLIGHT_DB, HOTEL_DB

USER_BOOKINGS = []

def search_flights(origin: str, destination: str):
    """Searches for flights and returns a high-level summary of available options.

    Args:
        origin: The origin city name of the flight, e.g., San Francisco.
        destination: The destination city name of the flight, e.g., New York.

    Returns:
        A list of available flights.
    """
    results = []
    for f in FLIGHT_DB:
        if (
            f["origin"].lower() == origin.lower()
            and f["destination"].lower() == destination.lower()
        ):
            results.append(
                {
                    "flight_id": f["flight_id"],
                    "airline": f["airline"],
                    "starting_price": f"${f['price']}",
                }
            )
    return results

def get_flight_details(flight_id: str):
    """Gets the full details, baggage policies, and class options for a specific flight."""
    for f in FLIGHT_DB:
        if f["flight_id"] == flight_id:
            return {"details": f}
    return {"error": "Flight not found."}

def book_flight(flight_id: str, seat_class: str):
    """Books a specific flight for a passenger in the chosen seat class.

    Args:
        flight_id: The unique identifier for the flight.
        seat_class: The chosen seat class (e.g., 'Economy', 'Business', 'First Class').

    Returns:
        A dictionary containing the booking status, booking ID, and total bookings, or an error message.
    """
    for f in FLIGHT_DB:
        if f["flight_id"] == flight_id:
            # Provide a default list if "class_options" is missing
            available_classes = f.get(
                "class_options",
                [
                    "economy",
                    "business",
                    "first class",
                    "Economy",
                    "Business",
                    "First Class",
                ],
            )
            if seat_class not in available_classes:
                return {
                    "error": f"Seat class '{seat_class}' not available. Options: {available_classes}"
                }

            booking_id = f"BKG-F-{uuid.uuid4().hex[:6]}"
            USER_BOOKINGS.append(
                {
                    "type": "flight",
                    "booking_id": booking_id,
                    "flight_id": flight_id,
                    "class": seat_class,
                }
            )
            return {
                "status": "Success",
                "booking_id": booking_id,
                "total_bookings": len(USER_BOOKINGS),
            }
    return {"error": "Flight ID not found."}

# --- HOTEL FUNCTIONS ---
def search_hotels(city: str, min_rating: float = 0.0):
    """Searches for hotels in a specific city, optionally filtering by minimum rating.

    Args:
        city: The name of the city to search for hotels.
        min_rating: The minimum acceptable rating for the hotel. Defaults to 0.0.

    Returns:
        A dictionary containing a list of hotels matching the search criteria.
    """
    results = []
    for h in HOTEL_DB:
        if h["city"].lower() == city.lower() and h["rating"] >= min_rating:
            results.append({"id": h["id"], "name": h["name"], "rating": h["rating"]})
    return {"hotels": results}

def get_hotel_details(hotel_id: str):
    """Gets detailed information for a hotel, including room types, pricing, and amenities.

    Args:
        hotel_id: The unique identifier for the hotel.

    Returns:
        A dictionary containing the detailed hotel information, or an error message if not found.
    """
    for h in HOTEL_DB:
        if h["id"] == hotel_id:
            return {"details": h}
    return {"error": "Hotel not found."}

def book_hotel(hotel_id: str, room_type: str):
    """Books a specific room type at a hotel for the given dates.

    Args:
        hotel_id: The unique identifier for the hotel.
        room_type: The type of room to book (e.g., 'Standard', 'Deluxe').

    Returns:
        A dictionary containing the booking status, booking ID, and total bookings, or an error message.
    """
    for h in HOTEL_DB:
        if h["id"] == hotel_id:
            if room_type not in h["room_types"]:
                return {
                    "error": f"Room type '{room_type}' not available. Options: {list(h['room_types'].keys())}"
                }
            booking_id = f"BKG-H-{uuid.uuid4().hex[:6]}"
            USER_BOOKINGS.append(
                {
                    "type": "hotel",
                    "booking_id": booking_id,
                    "hotel_id": hotel_id,
                    "room": room_type,
                }
            )
            return {
                "status": "Success",
                "booking_id": booking_id,
                "total_bookings": len(USER_BOOKINGS),
            }
    return {"error": "Hotel ID not found."}

# 1. The Domain Specialist for Flights
flight_agent = Agent(
    model="gemini-3.5-flash",
    name="flight_specialist",
    instruction="You are a flight booking specialist. Use search_flights to find IDs, then get_flight_details to verify luggage/refund policies, and book_flight when the user confirms.",
    tools=[search_flights, get_flight_details, book_flight],
)

# 2. The Domain Specialist for Hotels
hotel_agent = Agent(
    model="gemini-3.5-flash",
    name="hotel_specialist",
    instruction="You are a hotel booking specialist. Use search_hotels to find IDs, then get_hotel_details to compare room types and amenities, and finally book_hotel.",
    tools=[search_hotels, get_hotel_details, book_hotel],
)

# 3. The Root Orchestrator
#    name="travel_agent_v2" is what becomes the Agent Engine display_name.
#    A versioned name avoids colliding with deployments from earlier runs.
travel_agent = Agent(
    model="gemini-3.5-flash",  # Use a stronger reasoning model for orchestration
    name="travel_agent_v2",
    instruction="You are a primary travel concierge. Talk to the user to understand their full itinerary. Delegate flight-related tasks to the flight_specialist, and hotel-related tasks to the hotel_specialist. Synthesize their results to the user.",
    sub_agents=[flight_agent, hotel_agent],
)

### Deploy the agent to Agent Runtime (needed for Part 2)

Part 2 evaluates a *deployed* agent, so you deploy it here. Run the cell once: the deploy takes 5-10 minutes, creates a billable resource, and prints the deployed resource name when it finishes. If the kernel restarts later, re-run this cell. Skip this section if you only run Part 1.

In [7]:
# ============================================================
# DEPLOY  (run-once)
# Each deploy takes 5-10 min and creates a billable resource.
# If the kernel restarts, re-run this cell to redeploy.
# ============================================================
agent_engine = deploy_adk_agent(travel_agent, "us-central1")
print(f"Deployed: {agent_engine.api_resource.name}")


Deploying the agent to Agent Engine. This can take up to 10 mins.
Deployed: projects/545101063219/locations/us-central1/reasoningEngines/8103855344767205376


## Part 1: Evaluate the local agent with user simulation

In this part, the **agent runs locally in your notebook** and you drive the evaluation one step at a time: generate test scenarios, simulate the conversations, score them, and cluster the failures. The eval features are GEAP's (`client.evals`); only the agent and the User Simulator execute locally.

You use GEAP's predefined multi-turn metrics, two custom registered metrics, and Automatic Loss Analysis.


### Step 1: Generate test scenarios

You use `generate_conversation_scenarios` to create multi-turn test cases from the agent's description and a short instruction. Here the instruction asks for hard cases, such as unavailable destinations, repeated changes of mind, and invalid options, so the agent is likely to make mistakes you can analyze later.

**Notes:**
1. If you omit `model_name`, the service uses its default model. Setting `gemini-3.5-flash` keeps generation fast.
2. `count` sets how many scenarios you generate.
3. The cell may print warnings about optional modules (`botocore`, LiteLLM) and experimental features -- expected; you can ignore them.

> **Eval feature:** Steps 1 and 2 use the GEAP Gen AI Evaluation SDK (`client.evals`), not ADK's local `adk eval` framework. `generate_conversation_scenarios` is the platform's managed scenario-generation feature. (ADK's own build-time eval framework is the separate ADK lab.)


In [8]:
agent_info = types.evals.AgentInfo.load_from_agent(agent=travel_agent)

eval_dataset = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 7,
        "model_name": "gemini-3.5-flash",
        "generation_instruction": (
            "Generate adversarial scenarios that stress the agent's tool use and are "
            "likely to expose mistakes: the user asks to book flights to cities with no "
            "availability (for example Cairo or Reykjavik), changes the destination and "
            "dates several times in one conversation, requests seat classes or room types "
            "that do not exist, and pushes the agent to book before confirming a specific "
            "flight or hotel id. The user is impatient and gives incomplete details."
        ),
        "environment_context": (
            "Today is Monday. I am located in San Francisco. Flights and hotels are "
            "available to Paris, New York, Tokyo, Chicago, Sydney, London, Boston, "
            "Seattle, Miami, and Berlin. Cities such as Cairo and Reykjavik have no "
            "availability."
        ),
    },
    allow_cross_region_model=True,  # gemini-3.5-flash is preview-only in 'global'; route from us-central1
)
display(eval_dataset.eval_dataset_df.head())

# Show one example row in full (the table above truncates wide columns).
show_full_row(eval_dataset.eval_dataset_df, 0)


/opt/micromamba/lib/python3.12/site-packages/vertexai/_genai/types/evals.py:134: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  return FunctionTool(func=tool)._get_declaration()  # type: ignore[no-any-return]
14:27:01 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:27:01 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


,starting_prompt,conversation_plan
0,I need to travel from Boston. Can you get me a...,1. The user starts by requesting a flight from...
1,I need to book a flight from Los Angeles to Sy...,1. The user requests a flight from Los Angeles...
2,"Hi, I need to get out of San Francisco immedia...",The user starts by demanding an immediate trip...
3,I need a flight from San Francisco to Cairo de...,The user demands an immediate flight and hotel...
4,I need to get to Cairo immediately from San Fr...,The user starts by requesting a flight and hot...


FULL VIEW OF ROW 0 (all columns)

----- starting_prompt -----
I need to travel from Boston. Can you get me a flight to Reykjavik as soon as possible? I also need a hotel there with a 5-star rating.

----- conversation_plan -----
1. The user starts by requesting a flight from Boston to Reykjavik and a 5-star hotel in Reykjavik. 
2. If the agent reports there is no availability (or no options found) for flights or hotels in Reykjavik, the user will express annoyance and say, 'Fine, let's change the destination to New York instead.'
3. When asked about preferred class or room types for New York, the user will initially push back and refuse to give specific categories, saying, 'Just get me the cheapest flight and a basic room.'
4. Once the agent explains the available standard classes (e.g., Economy, Business, First Class) and room types (e.g., Standard, Deluxe), the user will agree to 'Economy' class for the flight and a 'Standard' room for the hotel.
5. When the agent presents specific f

### Step 2: Simulate the conversations

You run the agent against each scenario with a User Simulator. The simulator plays the user across several turns and records the full conversation trace, which you score in Step 3.

**Notes:**
1. `max_turn` caps the turns per conversation. Raise it if you see a turn-limit warning and want the complete trace.
2. The simulation takes a few minutes (7 conversations, several turns each); a progress bar tracks it, and the last line prints one conversation as a readable transcript.

> **Eval feature:** `run_inference` with `user_simulator_config` is GEAP's User Simulator (Gen AI Evaluation SDK), running here in **local mode**: the agent and the simulator both execute in your notebook and call Gemini for their turns. Part 2 runs the same simulation as a managed cloud job. This is not ADK's `adk eval` framework.


In [9]:
eval_dataset_with_trace = client.evals.run_inference(
    agent=travel_agent,
    src=eval_dataset,
    config={
        "user_simulator_config": {
            "max_turn": 4,
        }
    },
)
display(eval_dataset_with_trace.eval_dataset_df.head())

# Show one simulated conversation as a readable transcript (user, agent, tool calls).
show_conversation(eval_dataset_with_trace.eval_dataset_df, 0)


Local Agent Run:   0%|          | 0/7 [00:00<?, ?it/s]/opt/micromamba/lib/python3.12/site-packages/vertexai/_genai/_evals_common.py:1231: UserWarning: [EXPERIMENTAL] LlmBackedUserSimulator: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator = LlmBackedUserSimulator(
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/simulation/llm_backed_user_simulator.py:139: UserWarning: [EXPERIMENTAL] UserSimulator: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__(config, config_type=LlmBackedUserSimulator.config_type)
App "user_simulation_app" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cach

,starting_prompt,conversation_plan,agent_data
0,I need to travel from Boston. Can you get me a...,1. The user starts by requesting a flight from...,{'agents': {'travel_agent_v2': {'agent_id': 't...
1,I need to book a flight from Los Angeles to Sy...,1. The user requests a flight from Los Angeles...,{'agents': {'travel_agent_v2': {'agent_id': 't...
2,"Hi, I need to get out of San Francisco immedia...",The user starts by demanding an immediate trip...,{'agents': {'travel_agent_v2': {'agent_id': 't...
3,I need a flight from San Francisco to Cairo de...,The user demands an immediate flight and hotel...,{'agents': {'travel_agent_v2': {'agent_id': 't...
4,I need to get to Cairo immediately from San Fr...,The user starts by requesting a flight and hot...,{'agents': {'travel_agent_v2': {'agent_id': 't...


CONVERSATION (row 0)

[User] I need to travel from Boston. Can you get me a flight to Reykjavik as soon as possible? I also need a hotel there with a 5-star rating.
    -> tool call: transfer_to_agent({'agent_name': 'flight_specialist'})
    <- tool result: transfer_to_agent
    -> tool call: search_flights({'origin': 'Boston', 'destination': 'Reykjavik'})
    <- tool result: search_flights
    -> tool call: transfer_to_agent({'agent_name': 'hotel_specialist'})
    <- tool result: transfer_to_agent
    -> tool call: search_hotels({'min_rating': 5, 'city': 'Reykjavik'})
    <- tool result: search_hotels
    -> tool call: search_hotels({'city': 'Reykjavik'})
    <- tool result: search_hotels
    -> tool call: transfer_to_agent({'agent_name': 'travel_agent_v2'})
    <- tool result: transfer_to_agent

[agent] It looks like our initial searches for flights from Boston to Reykjavik and 5-star hotels in Reykjavik didn't return any direct results. 

To help me find the best options for you, co

### Step 3: Score the conversations

Now you score the simulated conversations. First you author and register two **custom metrics** (a computation metric and an LLM-judge metric), then you run `evaluate` with those plus GEAP's predefined multi-turn metrics. Scoring runs on the managed eval service.


#### 3.1 Create and register custom metrics

You write two custom metrics and register them with the platform so the evaluation service can run them: one computed, and one graded by an LLM judge.


##### 3.1.1 Author a computation metric for agent efficiency

To evaluate complex agent behaviors, you can define a custom computation-based metric using a Python function.

In the code below, you create a metric that evaluates the efficiency of the agent's multi-turn trajectory. The custom Python function parses the `agent_eval_data` to analyze the sequence of events and tool calls across all conversation turns.

Specifically, this efficiency metric calculates a score from 0.0 to 1.0 based on the following logic:
* It starts with a base score of 1.0.
* It applies a small penalty for every tool call made by the agent (encouraging the agent to reach the goal with fewer steps).
* It applies a heavier penalty for redundant or looping tool calls (e.g., calling the same function with the exact same arguments multiple times).

You define the logic as a Python string, package it into a metric object, and register it with the metric registry service.

In [10]:
efficiency_metric_code = """
import json

def evaluate(instance: dict) -> float:
    agent_data = instance.get('agent_eval_data', {})
    turns = agent_data.get('turns', [])

    total_calls = 0
    seen_calls = set()
    redundant_calls = 0

    for turn in turns:
        for event in turn.get('events', []):
            content = event.get('content', event)
            if content.get('role') == 'model':
                for part in content.get('parts', []):
                    fc = part.get('function_call')
                    if fc:
                        total_calls += 1
                        # Create a unique key for the call (name + args) to detect loops
                        call_key = (fc.get('name'), json.dumps(fc.get('args'), sort_keys=True))
                        if call_key in seen_calls:
                            redundant_calls += 1
                        seen_calls.add(call_key)

    # Demo Scoring Logic:
    # 1. Base Score: 1.0
    # 2. Penalty: -0.02 for every internal function call
    # 3. Penalty: -0.10 for every redundant call (looping behavior)

    score = 1.0 - (total_calls * 0.02) - (redundant_calls * 0.10)

    # Ensure score stays between 0 and 1
    return max(0.0, min(1.0, score))
"""

efficiency_metric = types.CodeExecutionMetric(
    name="multi_turn_efficiency", custom_function=efficiency_metric_code
)

In [11]:
# ============================================================
# REGISTER METRIC  (run-once)
# Re-running creates a duplicate registry entry (harmless in this lab project).
# If the kernel restarts, re-run this cell so the metric variables exist.
# ============================================================
efficiency_metric_path = client.evals.create_evaluation_metric(metric=efficiency_metric)
print("Metric has been registered at:", efficiency_metric_path)

registered_efficiency_metric = types.Metric(
    name="multi_turn_efficiency", metric_resource_name=efficiency_metric_path
)

Metric has been registered at: projects/545101063219/locations/us-central1/evaluationMetrics/67290111619891200


##### 3.1.2 Author an LLM-based metric for response tone

In addition to computation-based metrics, you can also define custom LLM-based metrics using `LLMMetric` object. You use an LLM "judge" to evaluate qualitative aspects of an agent's performance, such as tone or helpfulness.

In the example below, you author a metric to evaluate the agent's tone. The `LLMMetric` is configured with:
* **Instructions and Criteria**: A prompt template that defines specific dimensions -- Professionalism and Empathy -- for the judge to evaluate.
* **Input Variables**: Placeholders like `{agent_data}` that the service populates with the actual conversation trace during evaluation.
* **Structured Output Format**: Instructions for the LLM judge to return a JSON list of objects containing verdicts and reasoning for each property.
* **Custom Result Parsing**: A Python function that extracts the judge's JSON output, calculates a final numerical score based on the verdicts, and consolidates the reasoning into a single explanation.


In [12]:
tone_check_metric = types.LLMMetric(
    name="tone_check",
    prompt_template="""Analyze the tone of the response based on these two criteria:\n
          1. Professionalism: The response should use appropriate language and maintain a business-like demeanor.\n
          2. Empathy: The response should acknowledge the user's feelings and show understanding.\n\n
          Prompt: {agent_data.turns[0].events[0]}
          Response: {agent_data.turns[0].events[1]}
          Return ONLY a JSON list of objects for these two properties:
          [{"property": "Professionalism", "verdict": true, "reasoning": "..."},
          {"property": "Empathy", "verdict": true, "reasoning": "..."}]
        """,
    result_parsing_function="""
          import json, re
          def parse_results(responses):
              text = responses[0]
              # Use robust regex to find the JSON list block
              match = re.search("[\\[].*[]]", text, re.DOTALL)
              if not match: return {"score": 0.0, "explanation": "No valid JSON found"}

              try:
                  data = json.loads(match.group(0))
                  # Calculate an overall score (e.g., average of verdicts)
                  passed_count = sum(1 for r in data if r.get("verdict", False))
                  total_count = len(data)
                  score = passed_count / total_count if total_count > 0 else 0.0

                  # Consolidate reasoning into a single explanation string
                  explanation = "\\n".join([f"{r.get(\'property\')}: {r.get(\'reasoning\')}" for r in data])

                  # IMPORTANT: Return a dictionary, not a list
                  return {
                      "score": float(score),
                      "explanation": explanation
                  }
              except Exception as e:
                  return {"score": 0.0, "explanation": f"Parsing failed: {str(e)}"}
          """,
)


In [13]:
# ============================================================
# REGISTER METRIC  (run-once)
# Re-running creates a duplicate registry entry (harmless in this lab project).
# If the kernel restarts, re-run this cell so the metric variables exist.
# ============================================================
tone_check_metric_path = client.evals.create_evaluation_metric(metric=tone_check_metric)
print("Metric has been registered at:", tone_check_metric_path)

registered_tone_check_metric = types.Metric(
    name="tone-check", metric_resource_name=tone_check_metric_path
)

Metric has been registered at: projects/545101063219/locations/us-central1/evaluationMetrics/4552312390527483904


#### 3.2 Evaluate with predefined and custom metrics

You run `evaluate` with your two custom metrics plus GEAP's predefined multi-turn metrics for tool-use quality and task success. The call lowers the judge model's QPS: training projects have limited model quota, and the default rate can exceed it on long multi-turn traces.

The run takes about five minutes; the progress bar counts 7 cases x 4 metrics = 28 computations. You may see a few `Retryable error (code=503)` lines along the way -- the SDK retries automatically and the run continues.

**Note**: single-turn metrics do not work with simulated multi-turn agent_data. If you wish to use single-turn metrics like `GENERAL_QUALITY` or `SAFETY`, keep `max_turn` at 1 in `user_simulator_config` in the previous step.

In [16]:
eval_metrics = [
    registered_efficiency_metric,
    registered_tone_check_metric,
    types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
    types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
]

# Throttle calls to the judge model: training-environment projects have limited
# model quota, so the default evaluation_service_qps  can exceed it. A lower
# QPS keeps the judge calls within the training quota.
eval_result = client.evals.evaluate(
    dataset=eval_dataset_with_trace,
    metrics=eval_metrics,
    config=types.EvaluateMethodConfig(evaluation_service_qps=2),
)

Computing Metrics for Evaluation Dataset:  82%|████████▏ | 23/28 [02:43<01:31, 18.32s/it]Retryable error (code=503) on attempt 1/5 for metric 'multi_turn_tool_use_quality_v1': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Call to judge model deadline exceeded.', 'status': 'UNAVAILABLE'}}. Retrying in 1.8 seconds...
Retryable error (code=503) on attempt 1/5 for metric 'multi_turn_tool_use_quality_v1': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Call to judge model deadline exceeded.', 'status': 'UNAVAILABLE'}}. Retrying in 1.6 seconds...
Computing Metrics for Evaluation Dataset:  96%|█████████▋| 27/28 [05:27<00:50, 50.98s/it]Retryable error (code=503) on attempt 2/5 for metric 'multi_turn_tool_use_quality_v1': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Call to judge model deadline exceeded.', 'status': 'UNAVAILABLE'}}. Retrying in 2.8 seconds...
Retryable error (code=503) on attempt 2/5 for metric 'multi_turn_tool_use_quality_v1': 503 UNAVAILABLE. {'error': {'c

#### Read the results

The summary table shows one row per metric (`mean_score`, `pass_rate`, errors); the per-case table shows one row per conversation. Expect **low scores on the two custom metrics**: the scenarios are deliberately adversarial (unavailable cities, repeated changes of mind), and the efficiency metric penalizes every extra tool call those users force. The predefined metrics usually score higher -- the agent handles the chaos but takes extra steps. Read the comparison across metrics and cases rather than expecting high absolute scores.

In [17]:
# Build clean tables from the structured results: summary metrics and per-case scores.
import pandas as pd

# 1) Summary: one row per metric.
summary_rows = []
for m in (eval_result.summary_metrics or []):
    d = m.model_dump() if hasattr(m, "model_dump") else m
    summary_rows.append({
        "metric": d.get("metric_name"),
        "mean_score": d.get("mean_score"),
        "stdev": d.get("stdev_score"),
        "pass_rate": d.get("pass_rate"),
        "cases": d.get("num_cases_total"),
        "errors": d.get("num_cases_error"),
    })
print("=== Summary metrics (one row per metric) ===")
display(pd.DataFrame(summary_rows).round(3))

# 2) Per-case scores: one row per case, one column per metric.
per_case = []
for case in (eval_result.eval_case_results or []):
    d = case.model_dump() if hasattr(case, "model_dump") else case
    row = {"case": d.get("eval_case_index")}
    for cand in (d.get("response_candidate_results") or []):
        for name, res in (cand.get("metric_results") or {}).items():
            row[name] = (res or {}).get("score")
    per_case.append(row)
print(f"\n=== Per-case scores ({len(per_case)} cases) ===")
display(pd.DataFrame(per_case).round(3))


=== Summary metrics (one row per metric) ===


,metric,mean_score,stdev,pass_rate,cases,errors
0,multi_turn_efficiency,0.180,0.230,0.000,7,0
1,tone-check,0.286,0.267,0.000,7,0
2,multi_turn_tool_use_quality_v1,0.815,0.118,0.143,7,0
3,multi_turn_task_success_v1,0.589,0.248,0.000,7,0



=== Per-case scores (7 cases) ===


,case,multi_turn_efficiency,tone-check,multi_turn_tool_use_quality_v1,multi_turn_task_success_v1
0,0,0.00,0.0,0.704,0.333
1,1,0.00,0.5,0.676,0.833
2,2,0.46,0.5,0.750,0.333
3,3,0.00,0.0,1.000,0.667
4,4,0.00,0.5,0.875,0.875
5,5,0.32,0.5,0.909,0.750
6,6,0.48,0.0,0.794,0.333


### Step 4: Find failure clusters with loss analysis

You group the failing conversations into clusters with `generate_loss_clusters`. It reads the traces that scored as failures and sorts them into named failure categories, so you see the common ways the agent went wrong instead of just a low score.

**Notes:**
1. Loss analysis works with two predefined metrics, `MULTI_TURN_TASK_SUCCESS` and `MULTI_TURN_TOOL_USE_QUALITY`. Other metrics may not produce clusters.
2. If there are no failing cases to group, the result is empty. The adversarial scenarios from Step 1 are meant to produce failures, so you get clusters here.


In [18]:
# generate_loss_clusters is only available in the 'global' region,
# so use a dedicated client (our main `client` lives in us-central1).
global_client = Client(
    project=PROJECT_ID,
    location="global",
    http_options={"api_version": "v1beta1"},
)

loss_analysis = global_client.evals.generate_loss_clusters(
    eval_result=eval_result,
    metric=types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
)

# Print the loss-analysis clusters as structured text so they read clearly inline.
import json

results = loss_analysis.results or []
print(f"Loss analysis completed! {len(results)} result group(s).\n")

n_clusters = 0
for r in results:
    rd = r.model_dump() if hasattr(r, "model_dump") else r
    if isinstance(rd, dict) and "clusters" in rd:
        clusters = rd.get("clusters") or []
        metric = (rd.get("config") or {}).get("metric")
        print(f"Metric: {metric} | {len(clusters)} cluster(s)")
        for c in clusters:
            n_clusters += 1
            tax = c.get("taxonomy_entry") or {}
            print(
                f"  [{c.get('item_count', '?')} items] "
                f"{tax.get('l1_category', '?')} / {tax.get('l2_category', '')}"
            )
            if tax.get("description"):
                print(f"        {tax['description']}")
    else:                                   # truly unexpected shape: show it whole
        print(json.dumps(rd, indent=2, default=str, ensure_ascii=False))

if n_clusters == 0:
    print(
        "\nNo loss clusters were produced. This is expected when there are no failing "
        "cases to cluster, or the failures could not be categorized ('Other NA'). "
        "Loss analysis clusters failures for the MULTI_TURN_TASK_SUCCESS and "
        "MULTI_TURN_TOOL_USE_QUALITY metrics only. To surface clusters, run more or "
        "harder scenarios (raise `count`, or a trickier `generation_instruction`)."
    )


Loss analysis completed! 1 result group(s).

Metric: multi_turn_tool_use_quality_v1 | 2 cluster(s)
  [6 items] Tool Calling / Omission of Required Tool Call
        The agent fails to execute a necessary function. This encompasses two scenarios: 1) The agent attempts to answer directly without using any tools. 2) The agent skips a prerequisite tool in a sequential workflow (e.g., looking up an ID or contact info) and immediately calls a dependent function using guessed or hallucinated parameter values.
  [1 items] Tool Calling / Incorrect Parameter Value
        The agent correctly identifies the function and parameter name but provides a parameter value that is logically or factually incorrect, or fails to apply necessary data transformation. This includes using a value that contradicts the user's request, failing to convert units (e.g., minutes to seconds), or providing a logically flawed value.


## Part 2: Evaluate the deployed agent with a managed run

In this part, you evaluate the **agent deployed to Agent Runtime**, the way you would assess a production service. Instead of running the steps yourself, you hand the scenarios, the deployed agent, and the metrics to the **Eval Management Service**, which runs the simulated conversations, scores them, and performs loss analysis in **one managed server-side job**.

It is the same GEAP eval toolkit as Part 1, switched from local execution to a fully managed run.


### Step 1: Generate synthetic user scenarios

This is the same `generate_conversation_scenarios` call as Part 1, Step 1. You regenerate the scenarios so this part stands on its own; you could instead reuse the `eval_dataset` from Part 1.


In [19]:
agent_info = types.evals.AgentInfo.load_from_agent(agent=travel_agent)

eval_dataset = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 7,
        "model_name": "gemini-3.5-flash",
        "generation_instruction": (
            "Generate adversarial scenarios that stress the agent's tool use and are "
            "likely to expose mistakes: the user asks to book flights to cities with no "
            "availability (for example Cairo or Reykjavik), changes the destination and "
            "dates several times in one conversation, requests seat classes or room types "
            "that do not exist, and pushes the agent to book before confirming a specific "
            "flight or hotel id. The user is impatient and gives incomplete details."
        ),
        "environment_context": (
            "Today is Monday. I am located in San Francisco. Flights and hotels are "
            "available to Paris, New York, Tokyo, Chicago, Sydney, London, Boston, "
            "Seattle, Miami, and Berlin. Cities such as Cairo and Reykjavik have no "
            "availability."
        ),
    },
    allow_cross_region_model=True,  # gemini-3.5-flash is preview-only in 'global'; route from us-central1
)
display(eval_dataset.eval_dataset_df.head())

# Show one example row in full (the table above truncates wide columns).
show_full_row(eval_dataset.eval_dataset_df, 0)


,starting_prompt,conversation_plan
0,I need to get to Cairo immediately. Find me a ...,The user wants to travel from San Francisco (S...
1,Hello. I want to travel from San Francisco (SF...,The user wants a flight from SFO to Tokyo and ...
2,Our Reykjavik honeymoon is ruined because our ...,The user is highly emotional and wants a trip ...
3,I need SFO to Seattle flight options and top h...,The user wants Seattle options from SFO first....
4,I want to book a trip to Sydney from San Franc...,The user attempts to force an immediate bookin...


FULL VIEW OF ROW 0 (all columns)

----- starting_prompt -----
I need to get to Cairo immediately. Find me a flight leaving today from San Francisco (SFO) and a luxury hotel there.

----- conversation_plan -----
The user wants to travel from San Francisco (SFO) to Cairo and requires a luxury hotel. If the agent searches and reports that Cairo has no available flights or hotels, the user will express frustration and pivot to London, keeping SFO as the origin city. The user will demand a London flight and hotel but refuse to provide class or room preferences initially. Once the agent searches for London options and asks for preferences, the user will specify 'Business' class for the flight and 'Deluxe' room for the hotel. When the agent presents the search results, the user will select flight ID 'LHR-882' and hotel ID 'LND-SAVOY-12' and ask to hear their specific details. Once the details are explained, the user will confirm the booking of these specific options.


### Step 2: Run the managed evaluation

You trigger a remote evaluation job with the **Eval Management Service**. One managed run does three things:

1. **Runs the agent:** the service drives your deployed agent against the simulated users and collects the conversation traces. If you skipped the deploy step, set the `agent` parameter to an existing resource name, for example `projects/{project_number}/locations/{location}/reasoningEngines/{agent_engine_id}`.
2. **Scores the traces:** with the predefined metrics and your registered custom metrics.
3. **Clusters the failures:** loss analysis runs in the same job, on the metrics in `loss_analysis_metrics`.

**Notes:**
1. `max_turn: 4` caps the turns per simulated conversation.
2. The job bundles every step, so it takes several minutes (about seven in testing). While it creates the evaluation items you may see a stream of `ExperimentalWarning` lines -- expected. The poll below refreshes every few seconds and finishes with `Evaluation job completed in N seconds` followed by the evaluation report. For a faster run, use fewer metrics, a simpler `generation_instruction`, or drop loss analysis.

In [20]:
eval_metrics = [
    registered_efficiency_metric,
    registered_tone_check_metric,
    types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
    types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
]

loss_analysis_metrics = [
    types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
    types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
]

# Eval Run will run agent, generate traces, and then calculate the metrics.
eval_run = client.evals.create_evaluation_run(
    agent=agent_engine.api_resource.name,
    agent_info=agent_info,
    dataset=eval_dataset,
    metrics=eval_metrics,
    dest=EVAL_RUN_GCS_BUCKET,
    user_simulator_config={
        "max_turn": 4,
    },
    loss_analysis_metrics=loss_analysis_metrics,
    config=types.CreateEvaluationRunConfig(allow_cross_region_model=True),
)

In [21]:
poll_evaluation_run(eval_run, client)

Evaluation job completed in 713 seconds.


Failed to load evaluation result from GCS: gs://qwiklabs-gcp-04-8bc33a081848-agent-engine/result_6637339380023296000.json. Error: 1 validation error for EvaluationItemResult
request.candidateResponses.0.error
  Extra inputs are not permitted [type=extra_forbidden, input_value={'code': 13, 'message': "...has no attribute 'get'"}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
Failed to load evaluation result from GCS: gs://qwiklabs-gcp-04-8bc33a081848-agent-engine/result_7895057536896729088.json. Error: 1 validation error for EvaluationItemResult
request.candidateResponses.0.error
  Extra inputs are not permitted [type=extra_forbidden, input_value={'code': 13, 'message': "...has no attribute 'get'"}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


## Recap

You evaluated an agent with GEAP's managed eval tools, end to end:

- **Synthetic scenario generation** (`generate_conversation_scenarios`) created adversarial multi-turn test cases from the agent's description.
- **The User Simulator** (`run_inference` with `user_simulator_config`) played the user and recorded full conversation traces.
- **Predefined multi-turn metrics** plus **two custom registered metrics** (a computed efficiency score and an LLM-judged tone check) scored the traces.
- **Automatic Loss Analysis** (`generate_loss_clusters`) grouped the failures into named categories.
- **A managed evaluation run** (`create_evaluation_run`) ran simulation, scoring, and loss analysis server-side against the deployed agent.

Lab A (a separate notebook) covers the other half of the story: ADK's own build-time evaluation framework for the fast local loop.

### Learn more

- [Agent evaluation overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/agent-evaluation)
- [Evaluate agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)
- [Evaluate with simulated users](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)
- [Manage evaluation metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)
- [View evaluation results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results)